# Speaker Encoder — ECAPA-TDNN + GE2E Loss
**Google Colab T4 GPU | LibriSpeech test-clean | Resume-capable**

Что делает этот ноутбук:
1. Скачивает датасет LibriSpeech test-clean (400MB, 40 спикеров)
2. Обучает ECAPA-TDNN с GE2E loss — модель учится различать голоса
3. Каждые 5 эпох сохраняет чекпоинт на Google Drive
4. После истечения сессии — перезапускаешь и продолжаешь с того же места
5. В конце — t-SNE визуализация эмбеддингов

> Runtime → Change runtime type → **T4 GPU** → Save

In [ ]:
# ── CELL 1: Проверка GPU ──────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── CELL 2: Установка зависимостей ───────────────────────────────────────────
!pip install -q librosa soundfile einops tqdm matplotlib scikit-learn

In [ ]:
# ── CELL 3: Импорты ──────────────────────────────────────────────────────────
import os, random, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [ ]:
# ── CELL 4: Google Drive (чекпоинты сюда) ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/speaker_encoder'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
# ── CELL 5: Конфиг — меняй только здесь ─────────────────────────────────────
class Config:
    # Аудио
    sample_rate  = 16000
    n_mels       = 80
    n_fft        = 512
    hop_length   = 160    # 10ms
    win_length   = 400    # 25ms
    clip_seconds = 3.0    # длина случайного фрагмента

    # Модель
    C       = 512   # ширина каналов ECAPA-TDNN
    emb_dim = 256   # размер эмбеддинга голоса

    # Обучение
    N = 8     # спикеров в батче
    M = 8     # фраз на спикера в батче
    lr             = 1e-3
    epochs         = 300
    steps_per_epoch = 200  # шагов за эпоху
    save_every     = 5     # сохранять чекпоинт каждые N эпох

    # Пути
    data_path      = '/content/LibriSpeech/test-clean'
    checkpoint_dir = CHECKPOINT_DIR

cfg = Config()
print('Config OK')

In [ ]:
# ── CELL 6: Скачать датасет (LibriSpeech test-clean, 400MB) ──────────────────
import tarfile, urllib.request

def reporthook(count, block_size, total_size):
    pct = count * block_size * 100 // max(total_size, 1)
    print(f'\r  {pct}%', end='', flush=True)

if not Path(cfg.data_path).exists():
    url = 'https://www.openslr.org/resources/12/test-clean.tar.gz'
    dst = '/content/test-clean.tar.gz'
    print('Скачиваем LibriSpeech test-clean...')
    urllib.request.urlretrieve(url, dst, reporthook)
    print('\nРаспаковываем...')
    with tarfile.open(dst) as f:
        f.extractall('/content/')
    os.remove(dst)
    print('Готово!')
else:
    print('Датасет уже скачан')

speakers = [d for d in Path(cfg.data_path).iterdir() if d.is_dir()]
total_files = sum(len(list(s.rglob('*.flac'))) for s in speakers)
print(f'Спикеров: {len(speakers)}, файлов: {total_files}')

In [ ]:
# ── CELL 7: Dataset ───────────────────────────────────────────────────────────
class SpeakerDataset(Dataset):
    def __init__(self, data_path, cfg):
        self.cfg = cfg
        self.clip_samples = int(cfg.clip_seconds * cfg.sample_rate)
        self.speakers = {}

        for spk_dir in sorted(Path(data_path).iterdir()):
            if not spk_dir.is_dir():
                continue
            utts = list(spk_dir.rglob('*.flac')) + list(spk_dir.rglob('*.wav'))
            if len(utts) >= cfg.M:
                self.speakers[spk_dir.name] = [str(u) for u in utts]

        self.ids = list(self.speakers.keys())
        print(f'Валидных спикеров: {len(self.ids)}')

    def _load_clip(self, path):
        audio, _ = librosa.load(path, sr=self.cfg.sample_rate, mono=True)
        if len(audio) >= self.clip_samples:
            start = random.randint(0, len(audio) - self.clip_samples)
            audio = audio[start:start + self.clip_samples]
        else:
            audio = np.pad(audio, (0, self.clip_samples - len(audio)))
        return audio

    def _mel(self, audio):
        mel = librosa.feature.melspectrogram(
            y=audio, sr=self.cfg.sample_rate,
            n_fft=self.cfg.n_fft, hop_length=self.cfg.hop_length,
            win_length=self.cfg.win_length, n_mels=self.cfg.n_mels,
            fmin=80, fmax=7600
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        return (log_mel - log_mel.mean()) / (log_mel.std() + 1e-8)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        spk = self.ids[idx]
        paths = random.sample(self.speakers[spk], self.cfg.M)
        mels = np.stack([self._mel(self._load_clip(p)) for p in paths])
        return torch.tensor(mels), spk   # (M, n_mels, T)


class GE2ESampler:
    """Каждый шаг — N случайных спикеров"""
    def __init__(self, n_speakers, N, steps):
        self.n, self.N, self.steps = n_speakers, N, steps
    def __iter__(self):
        for _ in range(self.steps):
            yield random.sample(range(self.n), self.N)
    def __len__(self):
        return self.steps


def collate(batch):
    mels = torch.stack([b[0] for b in batch])  # (N, M, n_mels, T)
    spks = [b[1] for b in batch]
    return mels, spks


dataset = SpeakerDataset(cfg.data_path, cfg)
sampler = GE2ESampler(len(dataset), cfg.N, cfg.steps_per_epoch)
loader  = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate, num_workers=2)

In [ ]:
# ── CELL 8: Модель ECAPA-TDNN ─────────────────────────────────────────────────
class Res2Conv(nn.Module):
    def __init__(self, channels, kernel_size, dilation, scale=8):
        super().__init__()
        self.scale = scale
        w = channels // scale
        pad = dilation * (kernel_size - 1) // 2
        self.convs = nn.ModuleList([
            nn.Conv1d(w, w, kernel_size, padding=pad, dilation=dilation)
            for _ in range(scale - 1)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(w) for _ in range(scale - 1)])

    def forward(self, x):
        chunks = torch.chunk(x, self.scale, dim=1)
        out = [chunks[0]]
        for i in range(1, self.scale):
            y = chunks[i] if i == 1 else chunks[i] + out[-1]
            out.append(F.relu(self.bns[i-1](self.convs[i-1](y))))
        return torch.cat(out, dim=1)


class SEBlock(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // r), nn.ReLU(),
            nn.Linear(channels // r, channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x.mean(-1)).unsqueeze(-1)


class SERes2Block(nn.Module):
    def __init__(self, C, kernel_size, dilation):
        super().__init__()
        self.conv1 = nn.Conv1d(C, C, 1)
        self.bn1   = nn.BatchNorm1d(C)
        self.res2  = Res2Conv(C, kernel_size, dilation)
        self.conv2 = nn.Conv1d(C, C, 1)
        self.bn2   = nn.BatchNorm1d(C)
        self.se    = SEBlock(C)

    def forward(self, x):
        r = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.res2(x))
        x = self.se(self.bn2(self.conv2(x)))
        return F.relu(x + r)


class ASP(nn.Module):
    """Attentive Statistics Pooling"""
    def __init__(self, in_dim, attn_dim=128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_dim * 3, attn_dim, 1), nn.Tanh(),
            nn.Conv1d(attn_dim, in_dim, 1),     nn.Softmax(dim=-1)
        )

    def forward(self, x):
        mu  = x.mean(-1, keepdim=True).expand_as(x)
        std = x.std(-1,  keepdim=True).expand_as(x)
        a = self.attn(torch.cat([x, mu, std], 1))
        m = (a * x).sum(-1)
        s = torch.sqrt((a * x**2).sum(-1) - m**2 + 1e-8)
        return torch.cat([m, s], 1)


class ECAPA_TDNN(nn.Module):
    def __init__(self, C=512, emb_dim=256, n_mels=80):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(n_mels, C, 5, padding=2), nn.BatchNorm1d(C), nn.ReLU())
        self.b1 = SERes2Block(C, 3, 2)
        self.b2 = SERes2Block(C, 3, 3)
        self.b3 = SERes2Block(C, 3, 4)
        self.agg = nn.Sequential(nn.Conv1d(C*3, C*3, 1), nn.BatchNorm1d(C*3), nn.ReLU())
        self.asp = ASP(C*3)
        self.head = nn.Sequential(nn.BatchNorm1d(C*6), nn.Linear(C*6, emb_dim), nn.BatchNorm1d(emb_dim))

    def forward(self, x):
        x = self.stem(x)
        x1 = self.b1(x)
        x2 = self.b2(x + x1)
        x3 = self.b3(x + x1 + x2)
        x  = self.agg(torch.cat([x1, x2, x3], 1))
        x  = self.asp(x)
        x  = self.head(x)
        return F.normalize(x, dim=-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = ECAPA_TDNN(cfg.C, cfg.emb_dim, cfg.n_mels).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Параметров: {n_params:.1f}M | Device: {device}')

In [ ]:
# ── CELL 9: GE2E Loss ─────────────────────────────────────────────────────────
class GE2ELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(10.0))
        self.b = nn.Parameter(torch.tensor(-5.0))

    def forward(self, emb, N, M):
        # emb: (N*M, D) — L2 normalized
        emb = emb.view(N, M, -1)                     # (N, M, D)
        centroids = emb.mean(1)                       # (N, D)

        sim = torch.zeros(N, M, N, device=emb.device)
        for j in range(N):
            for i in range(N):
                if i == j:                            # centroid без текущей фразы
                    c = (emb[j].sum(0, keepdim=True) - emb[j]) / (M - 1)
                    sim[j, :, i] = F.cosine_similarity(emb[j], c)
                else:
                    sim[j, :, i] = F.cosine_similarity(
                        emb[j], centroids[i].unsqueeze(0).expand(M, -1))

        sim = self.w.abs() * sim + self.b             # масштаб обучаемый
        labels = torch.arange(N, device=emb.device).repeat_interleave(M)
        return F.cross_entropy(sim.view(N*M, N), labels)


criterion = GE2ELoss().to(device)
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(criterion.parameters()), lr=cfg.lr
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.epochs)
print('Loss & optimizer OK')

In [ ]:
# ── CELL 10: Чекпоинт: сохранение и загрузка ─────────────────────────────────
CKPT = os.path.join(cfg.checkpoint_dir, 'ecapa_ge2e.pt')
LOG  = os.path.join(cfg.checkpoint_dir, 'history.json')

def save(epoch, loss):
    torch.save({
        'epoch': epoch, 'loss': loss,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'criterion': criterion.state_dict(),
    }, CKPT)

def load():
    if not os.path.exists(CKPT):
        print('Чекпоинт не найден — обучаемся с нуля')
        return 0, []
    ckpt = torch.load(CKPT, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    criterion.load_state_dict(ckpt['criterion'])
    history = json.load(open(LOG)) if os.path.exists(LOG) else []
    print(f'Продолжаем с эпохи {ckpt["epoch"]}, loss={ckpt["loss"]:.4f}')
    return ckpt['epoch'], history

start_epoch, history = load()

In [ ]:
# ── CELL 11: Тренировочный цикл ───────────────────────────────────────────────
print(f'Обучение с эпохи {start_epoch+1} по {cfg.epochs}')
print(f'Батч: {cfg.N} спикеров × {cfg.M} фраз = {cfg.N*cfg.M} сэмплов')
print(f'Шагов в эпохе: {cfg.steps_per_epoch}')
print('─' * 50)

for epoch in range(start_epoch, cfg.epochs):
    model.train()
    losses = []

    pbar = tqdm(loader, desc=f'Epoch {epoch+1:3d}/{cfg.epochs}', leave=False)
    for mels, _ in pbar:
        N, M, n_mels, T = mels.shape
        flat = mels.view(N*M, n_mels, T).to(device)

        emb  = model(flat)
        loss = criterion(emb, N, M)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    avg = float(np.mean(losses))
    history.append({'epoch': epoch+1, 'loss': avg})

    print(f'Epoch {epoch+1:3d}/{cfg.epochs}  loss={avg:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  '
          f'w={criterion.w.item():.2f}  b={criterion.b.item():.2f}')

    if (epoch + 1) % cfg.save_every == 0:
        save(epoch + 1, avg)
        json.dump(history, open(LOG, 'w'))
        print(f'  ✓ сохранено в {CKPT}')

print('Обучение завершено!')

In [ ]:
# ── CELL 12: График loss ──────────────────────────────────────────────────────
epochs_plot = [h['epoch'] for h in history]
losses_plot = [h['loss']  for h in history]

plt.figure(figsize=(12, 4))
plt.plot(epochs_plot, losses_plot, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('GE2E Loss')
plt.title('Speaker Encoder Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.checkpoint_dir, 'training_curve.png'), dpi=150)
plt.show()
print(f'Финальный loss: {losses_plot[-1]:.4f}')

In [ ]:
# ── CELL 13: t-SNE — визуализация эмбеддингов ────────────────────────────────
# Если всё работает — точки одного спикера должны кластеризоваться вместе

@torch.no_grad()
def visualize(n_spk=10, n_utt=8):
    model.eval()
    embs, labels = [], []

    for i, spk in enumerate(dataset.ids[:n_spk]):
        paths = random.sample(dataset.speakers[spk], min(n_utt, len(dataset.speakers[spk])))
        for p in paths:
            audio = dataset._load_clip(p)
            mel   = dataset._mel(audio)
            t     = torch.tensor(mel).unsqueeze(0).to(device)
            embs.append(model(t).cpu().numpy()[0])
            labels.append(i)

    embs_2d = TSNE(n_components=2, random_state=42, perplexity=min(15, len(embs)-1)).fit_transform(np.array(embs))

    plt.figure(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, n_spk))
    for i in range(n_spk):
        mask = np.array(labels) == i
        plt.scatter(embs_2d[mask, 0], embs_2d[mask, 1],
                    color=colors[i], label=f'Spk {dataset.ids[i]}', s=120, alpha=0.85)
    plt.legend(fontsize=8, loc='best')
    plt.title('Speaker Embeddings — t-SNE\n(хорошо = чёткие кластеры, без перекрытий)')
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.checkpoint_dir, 'tsne.png'), dpi=150)
    plt.show()

visualize()

In [ ]:
# ── CELL 14: Демо — получить эмбеддинг для своего голоса ─────────────────────
# Загрузи свой WAV файл и получи 256-мерный эмбеддинг

from google.colab import files

@torch.no_grad()
def get_embedding(wav_path):
    model.eval()
    audio, _ = librosa.load(wav_path, sr=cfg.sample_rate, mono=True)
    # берём первые clip_seconds секунд
    clip = int(cfg.clip_seconds * cfg.sample_rate)
    if len(audio) > clip:
        audio = audio[:clip]
    else:
        audio = np.pad(audio, (0, clip - len(audio)))
    mel = dataset._mel(audio)
    t   = torch.tensor(mel).unsqueeze(0).to(device)
    emb = model(t).cpu().numpy()[0]
    print(f'Embedding shape: {emb.shape}')
    print(f'Norm: {np.linalg.norm(emb):.4f} (должна быть ~1.0)')
    return emb

print('Загрузи WAV файл кнопкой ниже:')
uploaded = files.upload()
for fname in uploaded:
    emb = get_embedding(fname)
    np.save(os.path.join(cfg.checkpoint_dir, f'{fname}.npy'), emb)
    print(f'Сохранён: {cfg.checkpoint_dir}/{fname}.npy')